In [0]:
import pyspark.sql.functions as F

In [0]:
df= spark.read.option("inferSchema", "true").json("abfss://bronze@deassessment.dfs.core.windows.net/payment_transaction")
df.display()
df.printSchema()

In [0]:
# Overwrite the column by casting it to "double"
df = df.withColumn("amount", F.col("amount").cast("double"))

df.printSchema()

In [0]:
df = df.withColumn(
    "payment_method",
    F.struct(
        F.struct(
            F.col("payment_method.details.bank").alias("bank"),
            F.col("payment_method.details.last_four").cast("int").alias("last_four")
        ).alias("details"),
        F.col("payment_method.type").alias("type")
    )
)

df.printSchema()

In [0]:
# Define the regex pattern: 
# ^ = start of string, [A-Za-z] = one alphabet, \d{6} = exactly 6 digits, $ = end of string
regex_pattern = r"^[L]\d{7}$"

# Filter for rows that DO NOT match the pattern
invalid_customers_df = df.filter(~F.col("loan_id").rlike(regex_pattern))

invalid_customers_df.display()

In [0]:
# EXTRA SAFE STEP: Force Spark to treat timezone-less timestamps as GMT
spark.conf.set("spark.sql.session.timeZone", "GMT")

# 1. Updated formats to precisely match your sample data
timestamp_formats = [
    "yyyy-MM-dd'T'HH:mm:ssXXX",     # Matches offsets like -08:00, -05:00, +00:00 (No milliseconds)
    "yyyy-MM-dd'T'HH:mm:ss",        # Matches the ones that stop at seconds (e.g., 2023-03-25T10:15:00)
    "yyyy-MM-dd HH:mm:ss",          # Standard space separator fallback
    "dd/MM/yyyy HH:mm:ss.SSS"       # Legacy fallback
]

# 2. Parse using F.lit() to avoid the column resolution error
parse_attempts = [F.try_to_timestamp(F.col("timestamp"), F.lit(fmt)) for fmt in timestamp_formats]

# 3. Coalesce to pick the first successful match
df_standard = df.withColumn("parsed_timestamp", F.coalesce(*parse_attempts))

# 4. Standardize everything to GMT/UTC
df = df_standard.drop("timestamp").withColumn("timestamp", F.to_utc_timestamp(F.col("parsed_timestamp"), "GMT")).drop("parsed_timestamp")

df.display()

In [0]:
df= df.distinct()

In [0]:
df.groupBy("payment_id").count().orderBy(F.col("count").desc()).display()

In [0]:
df.count()

In [0]:
df.select("payment_id").distinct().count()

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@deassessment.dfs.core.windows.net/payment/")